# Проверка решения без повторного обучения

Этот notebook заново строит поиск по всем объявлениям benchmark, рассчитывает
признаки и получает предсказания для всех запросов с сохранённой моделью.
Готовые индексы, признаки и ответы в расчёте не используются.

Нужны только `data/benchmark_items.parquet` и `data/benchmark_queries.parquet`.
Запустите все ячейки сверху вниз. Рабочие файлы попадут в `work/check/`.
Параметры поиска остаются такими же, как в полном эксперименте.

Обучение и измерение Recall доступны отдельно в `solution.ipynb`.
Здесь проверяется воспроизведение предсказаний, а не повторное обучение модели.

## 1. Настройки и модель

Проверяем сохранённую модель и код. `configs/check.toml` задаёт отдельные рабочие папки;
алгоритмические параметры должны совпадать с `configs/ranking.toml`.

In [1]:
import json
from pathlib import Path
import shutil
import subprocess
import sys
import time

from avito_ranker.config import load_config
from avito_ranker.freeze import check_frozen
from avito_ranker.workflow import run_stage
from avito_retrieval.common import file_hash, save_json

started = time.perf_counter()
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
source = load_config(project_dir / "configs/ranking.toml")
config_path = project_dir / "configs/check.toml"
config = load_config(config_path)
path_settings = {"config_path", "project_dir", "data_dir", "work_dir", "results_dir"}
if {key: value for key, value in source.items() if key not in path_settings} != {
    key: value for key, value in config.items() if key not in path_settings
}:
    raise ValueError("Параметры check.toml должны совпадать с ranking.toml")
if config["work_dir"] == source["work_dir"] or config["results_dir"] == source["results_dir"]:
    raise ValueError("Для проверки нужны отдельные рабочие папки")
check_frozen(source)
print("Код, модель и параметры проверены")

resource module not available on Windows


Код, модель и параметры проверены


## 2. Тесты

Проверки метрики, разбиения, кандидатов и формата CSV не требуют обучения.

In [2]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    check=True, cwd=source["project_dir"],
)
print("Тесты пройдены")

Тесты пройдены


## 3. Корпус benchmark

Сохраняем объявления в порядке исходных `item_id`. Используем тот же порядок
и те же поля, что в полном расчёте. `train.parquet` для этого запуска не нужен.

In [3]:
import polars as pl

for name in ["benchmark_items.parquet", "benchmark_queries.parquet"]:
    if not (config["data_dir"] / name).is_file():
        raise FileNotFoundError(f"Поместите {name} в {config['data_dir']}")

corpus = config["work_dir"] / "benchmark"
corpus.mkdir(parents=True, exist_ok=True)
config["results_dir"].mkdir(parents=True, exist_ok=True)
item_columns = ["item_id", "item_location_id", "item_category_id", "item_microcat_id"]
items = pl.read_parquet(config["data_dir"] / "benchmark_items.parquet", columns=item_columns)
items.sort("item_id").with_row_index("doc_id").write_parquet(corpus / "corpus_manifest.parquet")
shutil.copyfile(config["data_dir"] / "benchmark_items.parquet", corpus / "corpus.parquet")
for name in ["ranker.cbm", "selected.json"]:
    shutil.copyfile(source["results_dir"] / name, config["results_dir"] / name)
print(f"Объявлений в корпусе: {items.height:,}")

Объявлений в корпусе: 189,212


## 4. Поисковые индексы

`run_stage` вызывает код этапа из `avito_ranker`. Три BM25-индекса строятся
заново по заголовкам, описаниям и параметрам. Логи сохраняются в `work/check/results/logs/`.
Этапы выполняются последовательно в отдельных процессах, чтобы освобождать память.

In [4]:
run_stage("benchmark_indices", config_path)

Начат этап: benchmark_index_title


resource module not available on Windows
Tokenized title stemming True vocab 26204 tokens 869708 seconds 4.9
INDEX DONE {"field": "title", "stemming": true, "documents": 189212, "vocabulary": 26205, "tokens": 869708, "nnz": 851848, "seconds": 9.574296236038208, "k1": 1.5, "b": 0.75, "method": "lucene", "corpus_manifest_sha256": "b124e81efa4cfa951bfe8390f371098ea9808621105e016d26bffcddc9fb3822", "corpus_sha256": "193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899", "normalization": "NFKC lower ё→е; Unicode alphanumeric tokens; no stopword removal"}



Начат этап: benchmark_index_params


resource module not available on Windows
Tokenized params stemming True vocab 78736 tokens 27714318 seconds 63.3
INDEX DONE {"field": "params", "stemming": true, "documents": 189212, "vocabulary": 78737, "tokens": 27714318, "nnz": 11675448, "seconds": 77.89963984489441, "k1": 1.5, "b": 0.75, "method": "lucene", "corpus_manifest_sha256": "b124e81efa4cfa951bfe8390f371098ea9808621105e016d26bffcddc9fb3822", "corpus_sha256": "193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899", "normalization": "NFKC lower ё→е; Unicode alphanumeric tokens; no stopword removal"}



Начат этап: benchmark_index_description


resource module not available on Windows
Tokenized description stemming True vocab 375961 tokens 34746547 seconds 105.4
INDEX DONE {"field": "description", "stemming": true, "documents": 189212, "vocabulary": 375962, "tokens": 34746547, "nnz": 20899938, "seconds": 127.16986799240112, "k1": 1.5, "b": 0.75, "method": "lucene", "corpus_manifest_sha256": "b124e81efa4cfa951bfe8390f371098ea9808621105e016d26bffcddc9fb3822", "corpus_sha256": "193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899", "normalization": "NFKC lower ё→е; Unicode alphanumeric tokens; no stopword removal"}



Этап завершён: benchmark_indices 220.46 с


## 5. Кандидаты и признаки

Для каждого запроса выполняем тот же поиск и рассчитываем те же 31 признак,
которые использовались при подготовке отправленного ответа.

In [5]:
run_stage("benchmark_features", config_path)

Начат этап: benchmark_features


resource module not available on Windows
benchmark: 200/2452, 6 s
benchmark: 400/2452, 15 s
benchmark: 600/2452, 21 s
benchmark: 800/2452, 28 s
benchmark: 1000/2452, 33 s
benchmark: 1200/2452, 39 s
benchmark: 1400/2452, 44 s
benchmark: 1600/2452, 49 s
benchmark: 1800/2452, 54 s
benchmark: 2000/2452, 59 s
benchmark: 2200/2452, 64 s
benchmark: 2400/2452, 69 s
{"fold": "benchmark", "queries": 2452, "rows": 2092376, "features": ["title_score", "title_relative", "title_rr", "title_local_rr", "description_score", "description_relative", "description_rr", "description_local_rr", "params_score", "params_relative", "params_rr", "params_local_rr", "filter_score", "filter_relative", "rrf", "same_location", "same_category", "delivery", "query_words", "filter_words", "title_length", "title_coverage", "title_jaccard", "params_length", "params_coverage", "filter_coverage", "item_price", "item_rating", "item_rating_reviews_count", "item_is_phone_hidden", "item_is_message_forbidden"], "seconds": 70.712

Этап завершён: benchmark_features 76.82 с


## 6. Предсказания

Загружаем сохранённый CatBoost, выбираем 50 объявлений на запрос и проверяем
CSV по исходным запросам и корпусу. Новый ответ находится в `work/check/results/answer.csv`.

In [6]:
run_stage("predict", config_path)

Начат этап: predict


resource module not available on Windows
{'valid': True, 'rows': 2452, 'min_items': 50, 'max_items': 50, 'sha256': 'f4ffbb7029d6ad1bb53f97b88bceb76a0a7d2fe51d681a4d905a4c6d23e48cc5', 'model_sha256': 'bdec464db6daa2cd293ad3a630563d91e03482587a6521a6e65aca8852e18035'}



Этап завершён: predict 3.43 с


## 7. Сравнение с приложенным ответом

Только после получения новых предсказаний сравниваем CSV с ответом в корне
репозитория. Одинаковые SHA-256 означают побайтовое совпадение файлов.
Если ответы отличаются, ячейка завершится с ошибкой.

In [7]:
answer_path = config["results_dir"] / "answer.csv"
reference_path = source["project_dir"] / "answer.csv"
answer_hash = file_hash(answer_path)
reference_hash = file_hash(reference_path)
if answer_hash != reference_hash:
    raise ValueError("Новый answer.csv отличается от приложенного ответа")

submission = json.loads((config["results_dir"] / "submission_check.json").read_text("utf-8"))
report = {
    "status": "passed",
    "seconds": round(time.perf_counter() - started, 2),
    "queries": submission["rows"],
    "items": items.height,
    "model_retrained": False,
    "indices_and_features_rebuilt": True,
    "answer_identical": True,
    "answer_sha256": answer_hash,
    "model_sha256": file_hash(config["results_dir"] / "ranker.cbm"),
    "inputs": {name: file_hash(config["data_dir"] / name) for name in
               ["benchmark_items.parquet", "benchmark_queries.parquet"]},
}
save_json(config["results_dir"] / "check_report.json", report)
print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "status": "passed",
  "seconds": 304.34,
  "queries": 2452,
  "items": 189212,
  "model_retrained": false,
  "indices_and_features_rebuilt": true,
  "answer_identical": true,
  "answer_sha256": "f4ffbb7029d6ad1bb53f97b88bceb76a0a7d2fe51d681a4d905a4c6d23e48cc5",
  "model_sha256": "bdec464db6daa2cd293ad3a630563d91e03482587a6521a6e65aca8852e18035",
  "inputs": {
    "benchmark_items.parquet": "193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899",
    "benchmark_queries.parquet": "e49de4fb76f03979a4af96034817767d39ae5de9f94e254fed028243188dda06"
  }
}
